Plot line counts for a single repository over time. Collect the data first with:

    code-metrics repo-history https://github.com/lsst/daf_butler

In [ ]:
%matplotlib widget

In [ ]:
import matplotlib.pyplot as plt
from lsst.codemetrics.plotting import (
    CLOC_CPP_ALIASES,
    apply_aliases,
    load_repo,
    pivot,
    select,
    top_languages,
)

In [ ]:
name = "daf_butler"
frame = load_repo(name)
frame.head()

In [ ]:
# select() with no counter= uses whatever backend the file holds, and
# warns only if it holds several. Pass counter="cloc" to pick one.
chosen = select(frame)

# Fold cloc's header language into C++ so the two plot as one series.
# A no-op for backends that do not report that language.
folded = apply_aliases(chosen, CLOC_CPP_ALIASES)

# Any real repository reports enough languages to bury the plot under
# its own legend. Rank by the largest each language ever reached, so a
# subsystem that grew and was later removed still shows its rise and
# fall. Raise n to see more.
top = top_languages(folded, n=5)

code = pivot(top, value="code")
comment = pivot(top, value="comment")
code.tail()

In [ ]:
fig, ax = plt.subplots()
for language in code.columns:
    ax.plot(code.index, code[language], label=f"{language} code")
    ax.plot(comment.index, comment[language], label=f"{language} comment")
ax.set_title(f"Lines of {name} code and comments")
ax.set_ylim(bottom=0)
# Park the legend outside the axes so it can never cover the curves.
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0, fontsize="small")

In [ ]:
# Uncomment to save a PDF for publications.
# ax.set_title("")
# fig.savefig(f"{name}-lines.pdf", bbox_inches="tight")